In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

zip_path = "/content/drive/MyDrive/final_main_dataset.tsv"

if os.path.exists(zip_path):
    print("Found file:", zip_path)
else:
    print("File not found. Check path and filename.")

Found file: /content/drive/MyDrive/final_main_dataset.tsv


In [3]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=54e816fdfadab71f79d0c5d694ce50c5afd26565f9b501804d0afe22899fe36f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [5]:
import os
import re
import random
import tempfile
import logging
from typing import List, Tuple

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Evaluation metrics
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.chrf_score import sentence_chrf
from rouge_score import rouge_scorer
import math

# SentencePiece
try:
    import sentencepiece as spm
    HAS_SPM = True
except Exception:
    HAS_SPM = False

logging.basicConfig(level=logging.INFO)

# ==================== CONFIG ====================
class Config:
    D_MODEL = 512
    NHEAD = 8
    NUM_ENCODER_LAYERS = 4
    NUM_DECODER_LAYERS = 4
    DIM_FEEDFORWARD = 2048
    DROPOUT = 0.1
    BATCH_SIZE = 32
    LEARNING_RATE = 3e-4
    MAX_SEQ_LEN = 64
    VOCAB_SIZE = 16000
    EPOCHS = 20
    DATA_PATH = "drive/MyDrive/final_main_dataset.tsv"
    MODEL_SAVE_PATH = "urdu_single_stage_transformer.pth"
    SPM_MODEL = "urdu_spm.model"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()
print(f"Using device: {config.DEVICE}")

# ==================== URDU NORMALIZER ====================
class UrduTextNormalizer:
    @staticmethod
    def normalize(text: str) -> str:
        if not isinstance(text, str):
            return ""
        text = text.strip()
        text = re.sub(r'[آأإ]', 'ا', text)
        text = re.sub(r'ي', 'ی', text)
        text = re.sub(r'ھ', 'ہ', text)
        text = re.sub(r'ك', 'ک', text)
        text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

# ==================== TOKENIZER (SentencePiece ONLY) ====================
class UrduTokenizer:
    def __init__(self, sp_model_path=config.SPM_MODEL):
        if not HAS_SPM:
            raise RuntimeError("sentencepiece is required but not installed.")
        if not os.path.exists(sp_model_path):
            raise FileNotFoundError(f"SentencePiece model not found at: {sp_model_path}")

        self.sp_model_path = sp_model_path
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(sp_model_path)
        logging.info(f"Using SentencePiece subword tokenizer: {sp_model_path}")

    def train_sentencepiece(self, sentences: List[str], model_prefix: str, vocab_size: int = config.VOCAB_SIZE):
        tmp = tempfile.NamedTemporaryFile(delete=False, mode='w', encoding='utf-8')
        for s in sentences:
            tmp.write(s.replace('\n', ' ') + '\n')
        tmp.flush(); tmp.close()

        spm.SentencePieceTrainer.train(
            input=tmp.name,
            model_prefix=model_prefix,
            vocab_size=vocab_size,
            model_type='unigram'
        )

        self.sp_model_path = model_prefix + '.model'
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(self.spm_model_path)
        os.unlink(tmp.name)
        logging.info(f"Trained SentencePiece model saved to {self.sp_model_path}")

    def tokenize(self, text: str) -> List[str]:
        return self.sp.encode_as_pieces(text)

    def encode(self, text: str, add_special_tokens=False, max_len=None) -> List[int]:
        ids = self.sp.encode_as_ids(text)
        if add_special_tokens:
            ids = [self.cls_token_id] + ids + [self.sep_token_id]
        if max_len:
            ids = ids[:max_len]
            ids += [self.pad_token_id] * max(0, max_len - len(ids))
        return ids

    def decode(self, indices: List[int]) -> str:
        indices = [i for i in indices if i not in (self.pad_token_id, self.cls_token_id, self.sep_token_id)]
        return self.sp.decode(indices)

    @property
    def pad_token_id(self): return 0
    @property
    def cls_token_id(self): return 2
    @property
    def sep_token_id(self): return 3

    def __len__(self):
        return self.sp.get_piece_size()

# ==================== DATA LOADING ====================
def load_urdu_sentences(path: str) -> List[str]:
    normalizer = UrduTextNormalizer()
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, sep='\t', dtype=str, encoding='utf-8', on_bad_lines='skip')
            if 'sentence' in df.columns:
                sentences = df['sentence'].fillna('').astype(str).tolist()
            else:
                sentences = df.fillna('').astype(str).agg(' '.join, axis=1).tolist()
            sentences = [normalizer.normalize(s) for s in sentences if isinstance(s, str)]
            sentences = [s for s in sentences if 3 <= len(s) <= 300]
            logging.info(f"Loaded {len(sentences)} sentences from {path}")
            return sentences
        except Exception as e:
            logging.warning(f"Failed to read TSV: {e}")
    return [
        "اردو ایک خوبصورت زبان ہے",
        "السلام علیکم",
        "آپ کیسے ہیں؟",
        "میں ٹھیک ہوں شکریہ۔"
    ] * 200

def build_conversational_pairs(sentences: List[str]) -> List[Tuple[str, str]]:
    pairs = []
    for i in range(len(sentences) - 1):
        s = sentences[i]; r = sentences[i+1]
        pairs.append((s, r))
    logging.info(f"Built {len(pairs)} conversational pairs")
    return pairs

# ==================== DATASET ====================
class ChatDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str,str]], tokenizer: UrduTokenizer, max_len=config.MAX_SEQ_LEN):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src = self.tokenizer.encode(src_text, add_special_tokens=True, max_len=self.max_len)
        tgt = self.tokenizer.encode(tgt_text, add_special_tokens=True, max_len=self.max_len)
        decoder_in = tgt[:-1]
        target_out = tgt[1:]
        return torch.tensor(src), torch.tensor(decoder_in), torch.tensor(target_out)

# ==================== MODEL ====================
class Encoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.D_MODEL)
        self.pos_embedding = nn.Embedding(config.MAX_SEQ_LEN, config.D_MODEL)
        enc_layer = nn.TransformerEncoderLayer(config.D_MODEL, config.NHEAD, config.DIM_FEEDFORWARD, config.DROPOUT, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, config.NUM_ENCODER_LAYERS)
        self.norm = nn.LayerNorm(config.D_MODEL)
    def forward(self, x):
        pos = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(pos)
        x = self.transformer(x)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.D_MODEL)
        self.pos_embedding = nn.Embedding(config.MAX_SEQ_LEN, config.D_MODEL)
        dec_layer = nn.TransformerDecoderLayer(config.D_MODEL, config.NHEAD, config.DIM_FEEDFORWARD, config.DROPOUT, batch_first=True)
        self.transformer = nn.TransformerDecoder(dec_layer, config.NUM_DECODER_LAYERS)
        self.norm = nn.LayerNorm(config.D_MODEL)
        self.output = nn.Linear(config.D_MODEL, vocab_size)
    def forward(self, tgt, memory, tgt_mask=None):
        pos = torch.arange(0, tgt.size(1), device=tgt.device).unsqueeze(0)
        tgt = self.embedding(tgt) + self.pos_embedding(pos)
        out = self.transformer(tgt, memory, tgt_mask=tgt_mask)
        out = self.norm(out)
        return self.output(out)

# ==================== TRAIN + EVALUATION ====================
def evaluate_model(encoder, decoder, tokenizer, pairs):
    bleu_scores, rouge_scores, chrf_scores = [], [], []
    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    smooth = SmoothingFunction().method1

    for s, r in random.sample(pairs, min(100, len(pairs))):
        pred = generate_response(encoder, decoder, tokenizer, s)
        bleu = sentence_bleu([r.split()], pred.split(), smoothing_function=smooth)
        rougeL = rouge.score(r, pred)['rougeL'].fmeasure
        chrf = sentence_chrf(r, pred)
        bleu_scores.append(bleu); rouge_scores.append(rougeL); chrf_scores.append(chrf)

    bleu_avg = sum(bleu_scores)/len(bleu_scores)
    rouge_avg = sum(rouge_scores)/len(rouge_scores)
    chrf_avg = sum(chrf_scores)/len(chrf_scores)
    return bleu_avg, rouge_avg, chrf_avg

def compute_perplexity(loss):
    return math.exp(loss) if loss < 20 else float('inf')

def train_direct(pairs: List[Tuple[str,str]], tokenizer: UrduTokenizer):
    dataset = ChatDataset(pairs, tokenizer)
    loader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)
    vocab_len = len(tokenizer)
    encoder, decoder = Encoder(vocab_len).to(config.DEVICE), Decoder(vocab_len).to(config.DEVICE)
    optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    for epoch in range(1, config.EPOCHS+1):
        encoder.train(); decoder.train()
        total_loss = 0.0
        for src, dec_in, dec_out in tqdm(loader, desc=f"Epoch {epoch}/{config.EPOCHS}"):
            src, dec_in, dec_out = src.to(config.DEVICE), dec_in.to(config.DEVICE), dec_out.to(config.DEVICE)
            optimizer.zero_grad()
            memory = encoder(src)
            seq_len = dec_in.size(1)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(config.DEVICE)
            logits = decoder(dec_in, memory, tgt_mask)
            loss = criterion(logits.view(-1, logits.size(-1)), dec_out.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        ppl = compute_perplexity(avg_loss)
        print(f"Epoch {epoch} | Avg Loss: {avg_loss:.4f} | Perplexity: {ppl:.2f}")

        if epoch % 2 == 0:
            bleu, rougeL, chrf = evaluate_model(encoder, decoder, tokenizer, pairs)
            print(f"Evaluation after Epoch {epoch}: BLEU={bleu:.3f}, ROUGE-L={rougeL:.3f}, chrF={chrf:.3f}")

    torch.save({"encoder": encoder.state_dict(), "decoder": decoder.state_dict()}, config.MODEL_SAVE_PATH)
    print(f"Model saved to {config.MODEL_SAVE_PATH}")
    return encoder, decoder

# ==================== GENERATION ====================
def top_k_top_p_filtering_logits(logits, top_k=50, top_p=0.9):
    top_k = min(top_k, logits.size(-1))
    if top_k > 0:
        kth_val = torch.topk(logits, top_k)[0][-1]
        logits[logits < kth_val] = -float("inf")
    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = False
        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[indices_to_remove] = -float("inf")
    return logits

def generate_response(encoder, decoder, tokenizer, text, max_len=config.MAX_SEQ_LEN, top_k=50, top_p=0.9, temperature=0.7):
    encoder.eval(); decoder.eval()
    input_ids = tokenizer.encode(text, add_special_tokens=True, max_len=max_len)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(config.DEVICE)
    with torch.no_grad():
        memory = encoder(input_tensor)
        output_ids = [tokenizer.cls_token_id]
        for _ in range(max_len-1):
            tgt = torch.tensor([output_ids], dtype=torch.long).to(config.DEVICE)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(config.DEVICE)
            logits = decoder(tgt, memory, tgt_mask=tgt_mask)[0, -1, :] / temperature
            logits = top_k_top_p_filtering_logits(logits, top_k, top_p)
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            if next_token == tokenizer.sep_token_id or next_token == tokenizer.pad_token_id:
                break
            output_ids.append(next_token)
        return tokenizer.decode(output_ids[1:])

# ==================== MAIN ====================
if __name__ == "__main__":
    sentences = load_urdu_sentences(config.DATA_PATH)
    print(f"Total Urdu sentences loaded: {len(sentences)}")

    if not os.path.exists(config.SPM_MODEL):
        print("Training new SentencePiece model...")
        tmp_tokenizer = UrduTokenizer.__new__(UrduTokenizer)
        tmp_tokenizer.train_sentencepiece(sentences, model_prefix="urdu_spm", vocab_size=config.VOCAB_SIZE)

    tokenizer = UrduTokenizer(config.SPM_MODEL)
    print(f"vocab size: {len(tokenizer)}")

    pairs = build_conversational_pairs(sentences)
    print(f"Total conversational pairs: {len(pairs)}")

    print("Training Start: ")
    encoder, decoder = train_direct(pairs, tokenizer)

    print("\nChatbot ready! Type 'exit' to quit.")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            break
        reply = generate_response(encoder, decoder, tokenizer, user_input)
        print(f"Bot: {reply}\n")

Using device: cuda
Total Urdu sentences loaded: 19995
vocab size: 6918
Total conversational pairs: 19994
Training Start: 


Epoch 1/20: 100%|██████████| 625/625 [01:36<00:00,  6.48it/s]


Epoch 1 | Avg Loss: 6.2536 | Perplexity: 519.87


Epoch 2/20: 100%|██████████| 625/625 [01:36<00:00,  6.47it/s]


Epoch 2 | Avg Loss: 5.1322 | Perplexity: 169.39
Evaluation after Epoch 2: BLEU=0.007, ROUGE-L=0.000, chrF=0.078


...

Epoch 18 | Avg Loss: 1.0805 | Perplexity: 2.95
Evaluation after Epoch 18: BLEU=0.006, ROUGE-L=0.000, chrF=0.097


Epoch 19/20: 100%|██████████| 625/625 [01:35<00:00,  6.53it/s]


Epoch 19 | Avg Loss: 1.0731 | Perplexity: 2.92


Epoch 20/20: 100%|██████████| 625/625 [01:35<00:00,  6.52it/s]


Epoch 20 | Avg Loss: 1.0598 | Perplexity: 2.89
Evaluation after Epoch 20: BLEU=0.006, ROUGE-L=0.000, chrF=0.090
Model saved to urdu_single_stage_transformer.pth

Chatbot ready! Type 'exit' to quit.
You: آپ کا نام کیا ہے؟
Bot: اس

You: آج موسم کیسا ہے؟
Bot: ہم کیوں نہیں یہ کرتے؟

You: آپ کہاں رہتے ہیں؟
Bot: اس نے اس کا نام رکہا

You: کیا وقت ہوا ہے؟
Bot: تو اسے بہاری قیمت چکانا پڑتی ہے۔

You: exit
Goodbye!


In [13]:
# ==================== MAIN + GRADIO UI====================
if __name__ == "__main__":
    sentences = load_urdu_sentences(config.DATA_PATH)
    print(f"Total Urdu sentences loaded: {len(sentences)}")

    # Train SentencePiece model if missing
    if not os.path.exists(config.SPM_MODEL):
        print("Training new SentencePiece model...")
        tmp_tokenizer = UrduTokenizer.__new__(UrduTokenizer)
        tmp_tokenizer.train_sentencepiece(sentences, model_prefix="urdu_spm", vocab_size=config.VOCAB_SIZE)

    tokenizer = UrduTokenizer(config.SPM_MODEL)
    print(f"SentencePiece vocab size: {len(tokenizer)}")

    pairs = build_conversational_pairs(sentences)
    print(f"Total conversational pairs: {len(pairs)}")

    encoder, decoder = None, None

    # ==================== LOAD OR TRAIN ====================
    if os.path.exists(config.MODEL_SAVE_PATH):
        print(f"Loading saved model from: {config.MODEL_SAVE_PATH}")
        checkpoint = torch.load(config.MODEL_SAVE_PATH, map_location=config.DEVICE)

        encoder = Encoder(
            vocab_size=len(tokenizer)
        ).to(config.DEVICE)

        decoder = Decoder(
            vocab_size=len(tokenizer)
        ).to(config.DEVICE)

        encoder.load_state_dict(checkpoint['encoder'])
        decoder.load_state_dict(checkpoint['decoder'])
        print("Model loaded successfully — ready for chat!")
    else:
        print("No saved model found. Training from scratch...")
        encoder, decoder = train_direct(pairs, tokenizer)
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict()}, config.MODEL_SAVE_PATH)
        print(f"Model trained and saved to {config.MODEL_SAVE_PATH}")

    # ==================== GRADIO CHATBOT ====================
    import gradio as gr
    conversation_history = []

    def chat_interface(user_input: str):
        global conversation_history
        if not user_input.strip():
            return ""
        reply = generate_response(
            encoder, decoder, tokenizer, user_input,
            max_len=config.MAX_SEQ_LEN,
            top_k=0,   # Greedy decoding
            top_p=0.0,
            temperature=1.0
        )
        conversation_history.append((user_input, reply))
        display_history = ""
        for u, b in conversation_history[-10:]:
            display_history += f"{u}\n{b}\n\n"
        return display_history.strip()

    iface = gr.Interface(
        fn=chat_interface,
        inputs=[gr.Textbox(label="آپ کا پیغام", placeholder="اردو میں لکھیں...", lines=2)],
        outputs=[gr.Textbox(label="چیٹ بوٹ کے جوابات", lines=20)],
        title="Urdu Chatbot",
        description="اردو چیٹ بوٹ"
    )

    iface.launch(share=True)

Total Urdu sentences loaded: 19995
SentencePiece vocab size: 6918
Total conversational pairs: 19994
Loading saved model from: urdu_single_stage_transformer.pth
Model loaded successfully — ready for chat!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://966c8de198af48f6d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
